## Normalization + Multilayer Perceptrons in a Transformer

**SwiGLU comparison and implementation module at the bottom of notebook.**

---

Notebook implements a basic pipeline of normalization and MLP:

multi-head attention output $X:[B,T,d_{model}]$ as the input \
↓ \
RMSNorm across features in $d_{model}$ \
↓ \
Up-projection $[B,T,d_{ff}]$ \
↓ \
Apply nonlinear activation function \
↓ \
Down-projection back to $[B,T,d_{model}]$ \
↓ \
output $Y$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
batch_size = 1
sequence_length = 4
d_model = 8

x = torch.randn(
    batch_size,
    sequence_length,
    d_model
)

print("x shape:", x.shape)

## Computing Root Mean Square Normalization (RMSNorm)

RMSNorm and its older cousin LayerNorm both do the same thing. They are normalization methods that perform the normalizations across each token $T$ across the values/features in $d_{model}$.

---

**Layer Normalization (LayerNorm) centers and rescales activations using both the mean and variance.**

LayerNorm equation: \
$\mathrm{LayerNorm}(x_i) = \gamma_i \frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}} + \beta_i$

**Mean** of the features being normalized: \
$\mu = \frac{1}{d_{model}}\sum_{i=1}^{d_{model}} x_i$

**Variance** of said features: \
$\sigma^2 = \frac{1}{d_{model}}\sum_{i=1}^{d_{model}}(x_i-\mu)^2$

$i$: feature/dimension index for each token's vector represenation in $T$ \
$\epsilon$: tiny constant value to ensure denominator is never 0 \
$\gamma _i$: learned scale parameter for each feature position $i$ 

---

**Root Mean Square Normalization (RMSNorm) simplifies the normalization approach by only rescaling activations using the root mean square.**

RMSNorm equation: \
$\mathrm{RMSNorm}(x_i) = \gamma_i \frac{x_i}{\mathrm{RMS}(x)}$

Root mean square equation: \
$\mathrm{RMS}(x) = \sqrt{\epsilon + \frac{1}{d_{model}}\sum_{i=1}^{d_{model}}x_i^2}$

$i$: feature/dimension position \
$\epsilon$: tiny constant \
$\gamma _i$: learned scale parameter

Because of the simpler compute process, RMSNorm can speed up the **normalization operation itself** by 10-50%, depending on architecture and implementation.

However, normalization makes up only a small portion of a model's **total computation**, so at most the overall performance is usually marginal.

---

We'll start by computing the $\mathrm{RMS}(x) = \sqrt{\epsilon + \frac{1}{d_{model}}\sum_{i=1}^{d_{model}}x_i^2}$ manually.

In [ ]:
epsilon = 1e-6 # tiny value for numerical stability

rms = torch.sqrt( # root
    x.pow(2) # square
    .mean(dim=-1, keepdim=True) # mean (across last dim)
    + epsilon
)

print("RMS shape:", rms.shape)

The last dimension in the tensor should be $1$ as the code gets the average. RMS shape $[B, T, 1]$. \
Essentially there's one RMS value per token representation $T$.

---

The actual normalization $\frac{x_i}{\mathrm{RMS}(x)} = \hat{x}_i$ is now applied, meaning each feature $i$ in a token vector is divided by that token vector's RMS.

In [ ]:
normalized = x / rms

print("Normalized shape:", normalized.shape)

Lastly a **learned** scalar parameter $\gamma _i$ is multiplied with $\hat{x}_i$, for each $i$. The weights start at $1$ and the model independently adjusts them during training. \
$\gamma$ is a vector with length $d_{model}$, and its weights are learned during training just like $W_Q, W_K, W_V, W_O$, etc.

---

Sometimes a corresponding learned bias $\beta _i$ is added in the end, however using biases vary between models. Many modern decoder-only Transformers use RMSNorm without a learned bias $\beta$.

In [ ]:
weight = torch.ones(d_model) # initial feature weights all 1

# y = γ * x̂
output = normalized * weight # no bias

print("Output shape:", output.shape)

$y_i = \gamma_i \frac{x_i}{\mathrm{RMS}(x)}$ can all be wrapped in a reusable `RMSNorm()` module.

In [ ]:
class RMSNorm(nn.Module):
    """
    Root mean square normalization.
    """

    def __init__(self, d_model, epsilon=1e-6):
        super().__init__()

        self.epsilon = epsilon
        self.weight = nn.Parameter( # learnable weight parameter
            torch.ones(d_model) # initially all 1
        )

    def forward(self, x):
        rms = torch.sqrt( # root
            x.pow(2).mean( # squared mean
                dim=-1,
                keepdim=True # keep dim to enable broadcasting
            )
            + self.epsilon
        )
        
        # normalize
        normalized = x / rms

        # y = γ * x̂
        return normalized * self.weight

In [ ]:
norm = RMSNorm(d_model)

output = norm(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Learned scale (weight) shape:", norm.weight.shape)

## Important Model Decision

### Normalization Implementation

- LayerNorm normalizes using both the **mean** and **variance** of a token's feature
values.

- RMSNorm uses the **root mean square** instead, mainly rescaling the magnitude
without subtracting the mean.

Both keep tensor shape:
$[B, T, d_{model}] \rightarrow [B, T, d_{model}]$

---

**RMSNorm is simpler, commonly used in modern decoder-only LLMs, and is looking like the stronger candidate normalization method for our LLM.**

## Multilayer Perceptron Feed-Forward Network

Unlike traditional neural networks that tend to have many layers, an MLP block only contains 2 linear layers.

In essence, the architecture of a transformer shifts the responsibility of **deep representation learning** away from a single, massively deep neural network and spreads it across a **deep stack of transformer blocks**.

---

The **first** linear layer performs **up-projection**, mapping the input into a higher-dimensional space: \
$X:[B,T,d_{model}] \times W_\uparrow:[d_{model},d_{ff}] \rightarrow Z:[B,T,d_{ff}]$

$d_{ff}$'s dimension is typically much larger, with a common initial value being $d_{ff} = 4 \times d_{model}$. This lets the model learn deeper, more complex transformations of each token’s information, but also makes the FFN a major source of Transformer parameters and compute.

↓

A **nonlinear activation** function is then applied: \
$H = \sigma (Z)$, where $H:[B,T,d_{ff}]$

Activations such as ReLU, GELU, SiLU, SwiGLU, introduce **nonlinearity** so the FFN can learn more **complex feature transformations** than linear layers alone.

↓

The **second** layer performs **down-projection** to the expected model dimension, restoring activated representation to $[B,T,d_{model}]$: \
$H:[B,T,d_{ff}] \times W_\downarrow:[d_{ff},d_{model}] \rightarrow Y:[B,T,d_{model}]$

This restores the standard $d_{model}$ dimension, allowing the FFN output to be added back through the **residual connection**.



In [ ]:
d_model = 8
d_ff = 32 # # feedforward dimension, typically 4x the size of d_model

x = torch.randn(
    batch_size,
    sequence_length,
    d_model
)

$W_\uparrow$ and $W_\downarrow$ may include learned biases as well.

**Whether biases should be applied will be a decision made during the final stages of coding the LLM, possibly alongside attention biases.**

---

Up-projection $W_\uparrow:[d_{model},d_{ff}]$ and down-projection $W_\downarrow:[d_{ff},d_{model}]$. These matrices are **learned parameters** just like $W_Q, W_K, W_V, W_O$.

In [ ]:
up_proj = nn.Linear(
    d_model,
    d_ff,
    bias=False
)

down_proj = nn.Linear(
    d_ff,
    d_model,
    bias=False
)

Conceptually, **up-projection** gives each token representation a larger feature space in which the MLP can perform its nonlinear transformation.

In [ ]:
# [B,T,d_model] -> [B,T,d_ff]
hidden = up_proj(x)

print("Input shape:", x.shape)
print("After up-projection:", hidden.shape)

In this small practical example, GELU will serve as the activation function.

Mathematically accurate **GELU** equation, which involves calculating **erf**: \
$\mathrm{GELU}(x) = \frac{x}{2}\left(1 + \mathrm{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $\mathrm{erf}(x)=\frac{2}{\sqrt{\pi}}\int_0^x e^{-t^2}\,dt$

---

A very close **approximate GELU** is commonly used as it has a slight numerical deviation from exact GELU, but is computationally **simpler/faster**: \
$\mathrm{GELU}(x) \approx \frac{x}{2}\left(1 + \tanh\left[\sqrt{\frac{2}{\pi}}\left(x + 0.044715x^3\right)\right]\right)$

For typical neural-network training, the approximation's error is generally too small to meaningfully affect model quality.

More on which equation is used in the doc: \
https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.gelu.html

In [ ]:
# Apply nonlinear activation
activated = F.gelu(hidden)

print("After GELU:", activated.shape)

**Down-projection** restores each now-activated token's representation to its original input shape.

Conceptually, $W_\downarrow$ learns how to **combine/compress** useful activated $d_{ff}$ features back into $d_{model}$.

In [ ]:
# [B,T,d_ff] -> [B,T,d_model]
output = down_proj(activated)

print("Final output shape:", output.shape)

It is important to note:

- **Attention** mixes information between tokens.
- The **MLP** transforms each token’s features independently.

The MLP **does not** mix information between tokens, it transforms the features **within each** token’s vector representation.

---

All of this can be wrapped in a cute little example `FeedForward` module.

In [ ]:
class FeedForward(nn.Module):
    """
    Traditional Transformer feed-forward network.
    """

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        x = self.up_proj(x) # [B,T,d_model] -> [B,T,d_ff]
        x = F.gelu(x) # apply activation (GELU)
        x = self.down_proj(x) # [B,T,d_ff] -> [B,T,d_model]

        return x

In [ ]:
# up-projection -> apply activation -> down-projection
mlp = FeedForward(
    d_model=d_model,
    d_ff=d_ff
)

output = mlp(x)

print(output.shape)

### Sigmoid Linear Unit (SiLU)

SiLU or otherwise known as **Swish** is another nonlinear activation function.

Mathematical notation: \
$\mathrm{SiLU}(x)=x\cdot\sigma(x)$, where the sigmoid function is $\sigma(x)=\frac{1}{1+e^{-x}}$

---

ReLU is a hard gate, while GELU and SiLU are **smooth gates**.

Looking at GELU and SiLU:
- $\text{GELU}(x)=x\Phi(x)$, where $\phi$ is the gaussian-based function
- $\text{SiLU}(x)=x\sigma(x)$, where $\sigma$ is the sigmoid function

Both essentially have the form "$x \times \text{smooth gate}$".

**The two perform quite similarly, however *SiLU* is preferred in modern LLMs it works particularly well as the activation inside *gated FFNs*, especially *SwiGLU*.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-4, 2, 500)

# Rectified Linear Unit
relu = np.maximum(0, x)

# Gaussian Error Linear Unit
gelu = 0.5 * x * (
    1 + np.tanh(
        np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)
    )
)

# Sigmoid Linear Unit
silu = x / (1 + np.exp(-x))

# Plot
plt.figure(figsize=(12, 5))

plt.plot(x, relu, label="ReLU", color="red")
plt.plot(x, gelu, label="GELU", color="blue")
plt.plot(x, silu, label="SiLU", color="green")

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.grid(alpha=0.3)

plt.xlabel("Input (x)")
plt.ylabel("Output")
plt.title("ReLU vs. GELU vs. SiLU")
plt.legend()

plt.xlim(-4, 2)
plt.ylim(-0.5, 2)

plt.xticks(np.arange(-4, 2.1, 0.5))
plt.yticks(np.arange(-0.5, 2.1, 0.5))

plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)

plt.show()

As seen the graph above, ReLU has a **hard gate** meaning there's a sharp cutoff at $x=0$, where negative activations are essentially disabled.

GELU and SiLU instead have **smooth gates**, where negative values are **gradually** suppressed rather than abruptly zeroed out. \
SiLU suppresses negative activations **more gradually** than GELU, allowing stronger negative signals to remain active over a wider input range, however this difference doesn't make a significant impact.

As a side note: ReLU is essentially deprecated and isn't commonly implemented in modern Transformer architectures.

---

Below is a tiny comparison of the 2 **smooth gated activations**, where we can see how GELU supresses negative signals more aggressively.

In [ ]:
# Sample inout values
values = torch.tensor([
    -3.0,
    -1.0,
    0.0,
    1.0,
    3.0
])

# Nonlinear activations
silu_values = F.silu(values) # SiLU
gelu_values = F.gelu(values) # GELU

print("SiLU:", silu_values)
print("GELU:", gelu_values)

## Gated MLP

### Context

A traditional MLP has 2 projection matrices $W_\uparrow$ and $W_\downarrow$, and its pipeline is:

normalized input \
↓ \
up-projection \
↓ \
nonlinear activation \
↓ \
down-projection \
 \
 \
We saw this with code:
````python
x = self.up_proj(x)
x = F.gelu(x)
x = self.down_proj(x)
````

---

### How Gated MLPs Work

A **gated** MLP has 3 projections, $W_\uparrow, W_\downarrow$, and a **learned** gate projection $W_{gate}$. The input $X$ is **up-projected** into 2 **separate** representations using $W_\uparrow$ and $W_{gate}$.

The MLP creates $W_\uparrow$ and $W_{gate}$ with dimension $[d_{model}, d_{ff}]$. These 2 are then multiplied **elementwise** $\odot$ after $W_{gate}$ has been activated. \
The conceptual pipeline looks like:
````text
                    ┌─────→ gate pathway (W_gate) ────→ activation ──────┐
input x ────────────┤                                                ⊙ ──→ down projection W_down
                    └──────────────→ value pathway (W_up) ───────────────┘
````

The input projected with $W_\uparrow$ contains features/information, and is then gated by $\odot W_{gate}$. \
The input projected with $W_{gate}$ is transformed with an activation function to determine how strongly each feature from $W_\uparrow$ should contribute.

---

**Gating** is using a learned gate representation to **elementwise-scale** another representation, controlling which features are **emphasized** or **suppressed**.

The code implementing **SwiGLU** below will roughly be:

$X:[B,T,d_{model}]$

↓

$X:[B,T,d_{model}] \times W_\uparrow:[d_{model},d_{ff}] \rightarrow U:[B,T,d_{ff}]$ \
$\text{SiLU}(X \times W_{gate}) \rightarrow G:[B,T,d_{ff}]$

↓

multiply elementwise $Y = G \odot U$ \
or everything expanded $Y = \text{SiLU}(XW_{gate}) \odot (XW_\uparrow)$

↓

$Y:[B,T,d_{ff}] \times W_\downarrow:[d_{ff},d_{model}]$

↓

now-transformed output $O:[B,T,d_{model}]$

Full equation:
$\text{MLP}(X) = W_\downarrow [\text{SiLU}(XW_{gate}) \odot (XW_\uparrow)]$

In [ ]:
batch_size = 1
sequence_length = 4
d_model = 8

# Sample input
x = torch.randn(
    batch_size,
    sequence_length,
    d_model
)

$W_{gate},W_\uparrow:[d_{model,d_{ff}}]$

In [ ]:
# Feedforward dimension
d_ff = 32

# W_gate
gate_proj = nn.Linear(
    d_model,
    d_ff,
    bias=False
)

# W_up
value_proj = nn.Linear(
    d_model,
    d_ff,
    bias=False
)

$XW_{gate},XW_\uparrow:[B,T,d_{ff}]$

In [ ]:
gate = gate_proj(x) # XW_gate
value = value_proj(x) # XW_up

print("Gate shape:", gate.shape)
print("Value shape:", value.shape)

$\text{SiLU}(XW_{gate})$

Then multiply elementwise $\text{SiLU}(XW_{gate}) \odot (XW_\uparrow)$

In [ ]:
# Apply SiLU only to the gate
gate = F.silu(gate)

hidden = gate * value # multiply elementwise instead of matrix multiplication

print("Gated hidden shape:", hidden.shape)

$W_\downarrow \times [\text{SiLU}(XW_{gate}) \odot (XW_\uparrow)]$

The tensor shape is now back to the input dimensions.

In [ ]:
down_proj = nn.Linear(
    d_ff,
    d_model,
    bias=False
)

output = down_proj(hidden)

print("Output shape:", output.shape)

## Important Model Decision

### SwiGLU Parameter Tradeoffs

Ignoring biases, gated MLPs requires more parameters if we assume both activation methods use the same projection matrix dimension.

- A standard MLP has $\approx 2d_{model}d_{ff}$ parameters from $W_\uparrow:[d_{model}, d_{ff}]$ and $W_\downarrow:[d_{ff}, d_{model}]$
- Gated MLPs has $\approx 3d_{model}d_{ff}$ parameters from $W_\uparrow,W_{gate}:[d_{model}, d_{ff}]$, and $W_\downarrow:[d_{ff}, d_{model}]$

Since SwiGLU has 3 projection matrices, it has about **50% more** parameters than a traditional MLP like SiLU.

---

The natural remedy for $\text{SwiGLU} = \frac{3}{2} \times \text{SiLU}$ (parameter count) is **reducing the projection dimensions**.

If we the standard MLP's projection size is: \
$d_{ff} = 4 \times d_{model}$

Then its parameter count is: \
$2(d_{model})(4 \times d_{model}) = 8 \times d_{model}$

If we want SwiGLU's parameter count to match: \
$3d_{model}d_{ff} \approx 8 \times d_{model}$ \
↓ \
solving for $d_{ff}$, we get $d_{ff} = \frac{8}{3} \times d_{model}$

So for example if $d_{model} = 512$ and we assume $d_{ff} = 4 \times d_{model}$:

$d_{ff} = 2048$ and so the SwiGLU parameter-match is $512 \times \frac{8}{3} = 1365$

Note: in practice, that SwiGLU parameter target would be rounded to a **hardware-friendly** number, for example $1365 \approx 1440 \approx 1280$.

---

### SwiGLU Considerations

- **Model quality**: SwiGLU adds expressive feature gating compared with a standard GELU/SiLU FFN.

- **Parameters**: The extra gate projection adds parameters, so $d_{ff}$ is typically reduced from the standard $4d_{model}$ to roughly $\frac{8}{3}d_{model}$ to maintain a similar parameter budget.

- **Compute/memory**: The additional projection increases matrix-multiplication compute, while $d_{ff}$ directly affects intermediate activation memory. Using $\frac{8}{3}d_{model}$ also keeps compute near a standard $4d_{model}$ FFN.

- **Hardware**: In practice, $d_{ff}$ is rounded to a hardware-friendly multiple (e.g. 64 or 128) rather than using exactly $\frac{8}{3}d_{model}$.

- **Inference**: FFN matrix multiplications are a significant part of decoder inference, so $d_{ff}$ also affects CPU inference cost.

- **Bias**: Making `bias=False` keeps parameter calculations simpler, so whether to use biases is a separate architectural choice.


## Comparison

GELU vs SwiGLU parameter counts.

https://docs.pytorch.org/docs/2.14/generated/torch.numel.html

In [ ]:
def count_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
    )

In [ ]:
gelu_mlp = FeedForward(
    d_model=8,
    d_ff=32
)

print(
    "GELU MLP parameters:",
    count_parameters(gelu_mlp)
)

In [ ]:
class SwiGLU(nn.Module):
    """
    Gated feedforward network using SiLU activation.
    """

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        gate = F.silu(self.gate_proj(x)) # apply activation
        value = self.up_proj(x)
        hidden = gate * value # multiply elementwise

        return self.down_proj(hidden)

We can now see side-by-side the parameter count of the standard and gated MLP.

In [ ]:
gelu_mlp = FeedForward(
    d_model=8,
    d_ff=32
)

swiglu = SwiGLU(
    d_model=8,
    d_ff=32
)

print("GELU MLP parameters:", count_parameters(gelu_mlp))
print("SwiGLU parameters:", count_parameters(swiglu))

ratio = count_parameters(swiglu) / count_parameters(gelu_mlp)

print(f"SwiGLU has {ratio}x more parameters than GELU")

We can parameter-match SwiGLU to contain 512 paramters.

Since $d_{model} = 8$ in our tiny example: \
SwiGLU projection matrices should be $d_{ff} = \frac{8}{3} d_{model} = \frac{64}{3} \approx 21.3$-dimensional for an exact match.

Since dimension sizes can't contain decimals, $d_{ff}$ will be **rounded**.

In [ ]:
matched_d_ff = 21

matched_swiglu = SwiGLU(
    d_model=8,
    d_ff=matched_d_ff
)

print("SwiGLU with parameter-matched dimension:", count_parameters(matched_swiglu))

output = swiglu(x)

print("\nInput shape:", x.shape)
print("Output shape:", output.shape)

assert output.shape == x.shape

## Final Verdict

A Transformer block contains normalization **before** both MLP and attention.

RMSNorm rescales each token representation while preserving $[B, T, d_{model}]$, it is favored over LayerNorm because of its **computational efficiency**.

---

A standard MLP: \
$d_{model} \rightarrow d_{ff} \rightarrow activation \rightarrow d_{model}$

Tokens are transformed **independently**.

---

SwiGLU adds a second learned $W_{gate}$ projection:

$\text{SiLU}(X \cdot W_{gate}) \times (X \cdot W_\uparrow)$

This allows the network to learn both the **hidden features** and how **strongly**
those features should pass through.

---

At the same $d_{ff}$, SwiGLU has about **50% more** MLP parameters than a standard
two-projection MLP. 
Using roughly $8/3 \times d_{model}$ instead of $4 \times d_{model}$ can make the parameter and compute budgets more comparable.

**The exact normalization type, MLP type, and hidden dimension remain model architecture decisions (TBD).**